# Final test-set labeling — v3 per-tweet RAG and v4 batched RAG (`tm_agents_final_12`)

**Group 12** | Labels the 2,388 unlabeled tweets in `data/test.csv` with BOTH final LLM models:

- **v3** — per-tweet RAG, k=8 (best macro-F1 on validation: 0.8556) → `pred_12_llm_v3.csv`
- **v4** — batched RAG, Optuna best config (0.8375 at ~1/8 cost, lowest neutral leak) → `pred_12_llm_v4.csv`

Plus an agreement analysis between the two. Everything is cached and resumable.
**The submitted `pred_12.csv` is never touched.**

Cost: v3 ≈ 2,388 Azure calls (~25 min) · v4 ≈ 240 calls (~4 min).

## 0. Setup + Azure

In [1]:
import os, json, re, time, hashlib, random
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
DATA_DIR, CACHE_DIR, STORE_DIR = "data/", "cache_agents/", "chroma_trainstore"
LLM_CACHE_FILE = os.path.join(CACHE_DIR, "test_llm_cache.json")
LABEL_NAMES = {0: "bearish", 1: "bullish", 2: "neutral"}
NAME_TO_ID  = {v: k for k, v in LABEL_NAMES.items()}

KEY_PATH = next((p for p in ["../Lab6/Azure Open AI Key.txt", "../Lab 6/Azure Open AI Key.txt",
                             "Azure Open AI Key.txt"] if os.path.exists(p)),
                "../Lab6/Azure Open AI Key.txt")
from langchain_openai import AzureChatOpenAI
with open(KEY_PATH, encoding="utf-8") as f: key = f.read().strip()
llm = AzureChatOpenAI(temperature=0, model="ChatGPT",
                      azure_endpoint="https://novaimsplayground.openai.azure.com/",
                      openai_api_type="azure", api_key=key,
                      api_version="2024-02-15-preview")
print("LLM ready:", llm.model_name)

c:\Users\artem\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


LLM ready: ChatGPT


## 1. Data: test set + retrieval store

In [2]:
def read_csv_safe(path):
    for enc in ["utf-8", "latin-1"]:
        try:    return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError: continue
    raise ValueError(f"Cannot decode {path}")

test_df = read_csv_safe(DATA_DIR + "test.csv")
test_df.columns = [c.strip().lower() for c in test_df.columns]
TEST_TEXTS = [str(t) for t in test_df["text"].tolist()]
TEST_IDS   = test_df["id"].tolist()
N_TEST = len(TEST_TEXTS)
print(f"Test tweets: {N_TEST}")

Test tweets: 2388


In [3]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="thenlper/gte-small",
    model_kwargs={"device": "cpu"}, encode_kwargs={"normalize_embeddings": True})
# Same store the validated configs were tuned against (80% training split).
vstore = Chroma(persist_directory=STORE_DIR, embedding_function=embedding_model)
print(f"Chroma store loaded ({STORE_DIR}), {vstore._collection.count()} training tweets.")

TEST_EMB_FILE = os.path.join(CACHE_DIR, "v4_test_emb_gte.npy")
if os.path.exists(TEST_EMB_FILE):
    test_emb = np.load(TEST_EMB_FILE)
    assert test_emb.shape[0] == N_TEST
    print("Test embeddings loaded from cache:", test_emb.shape)
else:
    test_emb = np.array(embedding_model.embed_documents(TEST_TEXTS))
    np.save(TEST_EMB_FILE, test_emb)
    print("Test embeddings computed and cached:", test_emb.shape)

C:\Users\artem\AppData\Local\Temp\ipykernel_7096\432198141.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\artem\AppData\Local\Temp\ipykernel_7096\432198141.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vstore = Chroma(persist_directory=STORE_DIR, embedding_function=embedding_model)


Chroma store loaded (chroma_trainstore), 7634 training tweets.
Test embeddings computed and cached: (2388, 384)


## 2. Shared helpers (cached LLM calls, parsing)

In [4]:
def safe_json(text):
    if text is None: return {}
    text = re.sub(r"^```(?:json)?|```$", "", str(text).strip(), flags=re.MULTILINE).strip()
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m: return {}
    try: return json.loads(m.group(0))
    except Exception: return {}

def norm_label(s):
    if s is None: return None
    s = str(s).strip().lower()
    for name, idx in NAME_TO_ID.items():
        if name in s: return idx
    if s in {"0", "1", "2"}: return int(s)
    return None

_llm_cache = json.load(open(LLM_CACHE_FILE)) if os.path.exists(LLM_CACHE_FILE) else {}
_dirty = 0
def _invoke_cached(system, user, retries=4):
    global _dirty
    k = hashlib.sha256(f"final12||{system}||{user}".encode()).hexdigest()[:32]
    if k in _llm_cache: return _llm_cache[k]
    from langchain_core.messages import SystemMessage, HumanMessage
    for attempt in range(retries):
        try:
            out = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)]).content
            _llm_cache[k] = out; _dirty += 1
            if _dirty >= 20:
                json.dump(_llm_cache, open(LLM_CACHE_FILE, "w")); _dirty = 0
            return out
        except Exception:
            if attempt == retries - 1: raise
            time.sleep(2 ** attempt)

def flush_cache(): json.dump(_llm_cache, open(LLM_CACHE_FILE, "w"))
print("Helpers ready. Cached responses:", len(_llm_cache))

Helpers ready. Cached responses: 0


## 3. v3 prompts + classifier *(copied verbatim from the experiment notebooks — originals untouched)*

In [5]:
FEWSHOT = [
    ("$VIAC: Barrington Research cuts to Mkt Perform", "bearish"),
    ("$JE - Just Energy reports lower than expected Q3 EBITDA; revises guidance", "bearish"),
    ("Wayfair shares surge 37% as coronavirus drives sales of home decor", "bullish"),
    ("Oil trades at 3-month high", "bullish"),
    ("$PEP - PepsiCo declares $0.955 dividend", "neutral"),
    ("$VIPS - Vipshop announces business cooperation with SF Holding", "neutral"),
]
_fewshot_txt = "\n".join(f'  Tweet: "{t}"  ->  {lab}' for t, lab in FEWSHOT)
NEUTRAL_RULE = (
    "A factual statement, news headline, analyst/price-target or rating note, dividend/earnings "
    "figure, corporate announcement, or a question with no clear directional opinion is NEUTRAL. "
    "Only choose bearish or bullish when the author expresses a directional view, or the news is "
    "unambiguously good or bad for the asset's price. When genuinely unsure between neutral and a "
    "direction, choose NEUTRAL."
)
CLASSIFIER_SYSTEM_V2 = (
    "You are a financial-sentiment classifier for stock-market tweets. Label each tweet's sentiment "
    "toward the asset/market as exactly one of: bearish, bullish, neutral.\n"
    "- bearish = expects price to fall / negative outlook / clearly bad news for the asset\n"
    "- bullish = expects price to rise / positive outlook / clearly good news for the asset\n"
    "- neutral = factual / mixed / no directional view.\n"
    f"IMPORTANT RULE: {NEUTRAL_RULE}\n"
    "Examples:\n" + _fewshot_txt
)

K_SHOT = 8
def retrieve_examples(tweet, k=K_SHOT):
    docs = vstore.similarity_search(tweet, k=k + 2)
    out = []
    for d in docs:
        if d.page_content.strip() == tweet.strip(): continue
        out.append((d.page_content, LABEL_NAMES[int(d.metadata["label"])]))
        if len(out) >= k: break
    return out

def classify_rag(tweet):
    ex = retrieve_examples(tweet)
    shots = "\n".join(f'Tweet: "{t}"  ->  {lab}' for t, lab in ex)
    user = ("Below are the most similar tweets from our labeled training data, with their correct "
            "labels. Use the SAME labeling standard they imply (especially for what counts as "
            "neutral vs directional):\n"
            f"{shots}\n\n"
            f'Now classify THIS tweet:\n\"\"\"{tweet}\"\"\"\n'
            "Reply with ONLY one word: bearish, bullish or neutral.")
    return norm_label((_invoke_cached(CLASSIFIER_SYSTEM_V2, user) or "").strip())

## 4. v3 full test run (cached/resumable, ~2,388 calls)

In [6]:
from tqdm.auto import tqdm
CACHE_T3 = os.path.join(CACHE_DIR, "test_preds_v3_rag_k8.json")
t3 = json.load(open(CACHE_T3)) if os.path.exists(CACHE_T3) else {}
todo = [i for i in range(N_TEST) if str(i) not in t3]
print(f"v3 on test: cached {len(t3)} | to run {len(todo)}")
for idx in tqdm(todo, desc="v3 RAG on test"):
    p = classify_rag(TEST_TEXTS[idx])
    t3[str(idx)] = 2 if p is None else int(p)
    if len(t3) % 50 == 0: json.dump(t3, open(CACHE_T3, "w"))
json.dump(t3, open(CACHE_T3, "w")); flush_cache()
preds_v3 = {i: t3[str(i)] for i in range(N_TEST)}
print("v3 label distribution:", Counter(LABEL_NAMES[v] for v in preds_v3.values()))

v3 on test: cached 0 | to run 2388


v3 RAG on test:   0%|          | 0/2388 [00:00<?, ?it/s]

v3 label distribution: Counter({'neutral': 1443, 'bullish': 528, 'bearish': 417})


## 5. v4 prompts, grouping, retrieval *(verbatim from `tm_v4_optuna_12.ipynb`, generalized to the test set)*

In [7]:
NEUTRAL_RULE_V4 = NEUTRAL_RULE
CLASSIFIER_SYSTEM_V4 = (
    "You are a financial-sentiment classifier for stock-market tweets. Label each tweet's sentiment "
    "toward the asset/market as exactly one of: bearish, bullish, neutral.\n"
    "- bearish = expects price to fall / negative outlook / clearly bad news for the asset\n"
    "- bullish = expects price to rise / positive outlook / clearly good news for the asset\n"
    "- neutral = factual / mixed / no directional view.\n"
    f"IMPORTANT RULE: {NEUTRAL_RULE_V4}\n"
    "You will receive a SHARED EXAMPLE BLOCK of similar training tweets with their correct labels, "
    "followed by a JSON batch of tweets to classify. Use the SAME labeling standard the examples "
    "imply, especially for what counts as neutral vs directional.\n"
    "IMPORTANT: the batch was deliberately built from MUTUALLY SIMILAR tweets, so it is expected "
    "and perfectly normal that MOST OR ALL tweets in a batch share the same label. Do NOT try to "
    "balance or diversify labels across the batch. Judge each tweet on its own."
)

# Optuna best config — read from the experiment's results log, fallback to the known values
try:
    BEST_CFG = json.load(open(os.path.join(CACHE_DIR, "results_log_v4.json")))["_meta"]["best_config"]
except Exception:
    BEST_CFG = {"k": 8, "batch_size": 10, "max_shared_examples": 24,
                "retrieval_mode": "nearest", "example_order": "shuffled"}
print("v4 config:", BEST_CFG)

from sklearn.cluster import KMeans
def make_batches(n, batch_size, emb, grouping_seed=RANDOM_STATE):
    idx = np.arange(n)
    km = KMeans(n_clusters=max(1, round(n / batch_size)), random_state=grouping_seed, n_init=10)
    lab = km.fit_predict(emb)
    groups = defaultdict(list)
    for i, l in zip(idx, lab): groups[int(l)].append(int(i))
    batches = []
    for l, members in groups.items():
        if len(members) <= batch_size * 1.5:
            batches.append(members)
        else:
            c = km.cluster_centers_[l]
            members = sorted(members, key=lambda i: -float(emb[i] @ c))
            for j in range(0, len(members), batch_size):
                batches.append(members[j:j + batch_size])
    done = False
    while not done:
        done = True
        for bi, b in enumerate(batches):
            if len(b) < 3 and len(batches) > 1:
                cb = emb[b].mean(axis=0)
                best, best_sim = None, -2
                for bj, other in enumerate(batches):
                    if bj == bi: continue
                    sim = float(cb @ emb[other].mean(axis=0))
                    if sim > best_sim: best, best_sim = bj, sim
                batches[best].extend(b); batches.pop(bi); done = False
                break
    return batches

MAX_K = 12
_retr_cache = {}
def _retrieve_one(i):
    if i in _retr_cache: return _retr_cache[i]
    docs = vstore.similarity_search_by_vector(test_emb[i].tolist(), k=MAX_K + 2)
    out = [(d.page_content.strip(), LABEL_NAMES[int(d.metadata["label"])])
           for d in docs if d.page_content.strip() != TEST_TEXTS[i].strip()]
    _retr_cache[i] = out
    return out

def build_user_prompt(example_block, batch_payload):
    return ("SHARED EXAMPLES (similar labeled training tweets):\n" + example_block +
            "\n\nClassify EVERY tweet below. Return ONLY a JSON object mapping each id to its label "
            '("bearish", "bullish" or "neutral"), e.g. {"0":"neutral","1":"bullish"}.\n'
            + json.dumps(batch_payload, ensure_ascii=False))

def classify_batch_v4(batch, cfg, rng):
    seen = {}
    for i in batch:
        for rank, (txt, lab) in enumerate(_retrieve_one(i)[:cfg["k"]]):
            if txt not in seen or rank < seen[txt][1]:
                seen[txt] = (lab, rank)
    items = sorted(seen.items(), key=lambda kv: kv[1][1])[:cfg["max_shared_examples"]]
    if cfg["example_order"] == "shuffled":
        items = items[:]; rng.shuffle(items)
    block = "\n".join(f'Tweet: "{t}"  ->  {lab}' for t, (lab, _) in items)
    payload = {str(i): TEST_TEXTS[i] for i in batch}
    parsed = safe_json(_invoke_cached(CLASSIFIER_SYSTEM_V4, build_user_prompt(block, payload)))
    out = {}
    for i in batch:
        out[i] = norm_label(parsed.get(str(i))) if isinstance(parsed, dict) else None
        if out[i] is None:                      # repair: single-tweet RAG with v4 system
            single = "\n".join(f'Tweet: "{t}"  ->  {lab}' for t, lab in _retrieve_one(i)[:cfg["k"]])
            user = ("SHARED EXAMPLES (similar labeled training tweets):\n" + single +
                    f'\n\nNow classify THIS tweet:\n\"\"\"{TEST_TEXTS[i]}\"\"\"\n'
                    "Reply with ONLY one word: bearish, bullish or neutral.")
            out[i] = norm_label((_invoke_cached(CLASSIFIER_SYSTEM_V4, user) or "").strip())
        if out[i] is None: out[i] = 2
    return out

v4 config: {'k': 8, 'batch_size': 10, 'max_shared_examples': 24, 'retrieval_mode': 'nearest', 'example_order': 'shuffled'}


## 6. v4 full test run (cached/resumable, ~240 calls)

In [8]:
CACHE_T4 = os.path.join(CACHE_DIR, "test_preds_v4.json")
t4 = json.load(open(CACHE_T4)) if os.path.exists(CACHE_T4) else {}
rng = random.Random(RANDOM_STATE)
batches = make_batches(N_TEST, BEST_CFG["batch_size"], test_emb)
print(f"v4 on test: {len(batches)} batches, cached {len(t4)}")
for b in tqdm(batches, desc="v4 batched RAG on test"):
    if all(str(i) in t4 for i in b): continue
    got = classify_batch_v4(b, BEST_CFG, rng)
    for i, p in got.items(): t4[str(i)] = int(p)
    json.dump(t4, open(CACHE_T4, "w"))
json.dump(t4, open(CACHE_T4, "w")); flush_cache()
preds_v4 = {i: t4[str(i)] for i in range(N_TEST)}
print("v4 label distribution:", Counter(LABEL_NAMES[v] for v in preds_v4.values()))

v4 on test: 292 batches, cached 0


v4 batched RAG on test:   0%|          | 0/292 [00:00<?, ?it/s]

v4 label distribution: Counter({'neutral': 1585, 'bullish': 450, 'bearish': 353})


## 7. Agreement analysis v3 vs v4

In [9]:
agree = sum(1 for i in range(N_TEST) if preds_v3[i] == preds_v4[i])
print(f"Agreement: {agree}/{N_TEST} = {100*agree/N_TEST:.2f}%\n")
print("Cross-tab (rows = v3, cols = v4):")
ct = np.zeros((3, 3), dtype=int)
for i in range(N_TEST): ct[preds_v3[i], preds_v4[i]] += 1
print(pd.DataFrame(ct, index=[f"v3:{LABEL_NAMES[i]}" for i in range(3)],
                       columns=[f"v4:{LABEL_NAMES[i]}" for i in range(3)]))
dis = [i for i in range(N_TEST) if preds_v3[i] != preds_v4[i]][:8]
print("\nSample disagreements:")
for i in dis:
    print(f"  [{i}] v3={LABEL_NAMES[preds_v3[i]]:8s} v4={LABEL_NAMES[preds_v4[i]]:8s} | {TEST_TEXTS[i][:90]}")

Agreement: 2159/2388 = 90.41%

Cross-tab (rows = v3, cols = v4):
            v4:bearish  v4:bullish  v4:neutral
v3:bearish         336           3          78
v3:bullish           2         421         105
v3:neutral          15          26        1402

Sample disagreements:
  [5] v3=bullish  v4=neutral  | Marvell Technology (MRVL) Gains As Market Dips: What You Should Know - Nasdaq
  [7] v3=bearish  v4=neutral  | why macro funds are shutting down left and right https://t.co/gVYowOBmqe
  [15] v3=bullish  v4=neutral  | Chipotle highlights fast drive-thru lanes
  [38] v3=bullish  v4=neutral  | UPDATE 2-Crunch talks lead to Emirates order for 30 Boeing Dreamliners
  [44] v3=bullish  v4=neutral  | Encantos Partners With Award-Winning Chef Aliya LeeKong to Develop New Food-Inspired Presc
  [48] v3=neutral  v4=bearish  | Eurozone PMI Shows Economy Close To Stagnation. Sign up for updates on Seeking Alpha! http
  [57] v3=neutral  v4=bullish  | $UDIRF $UDIRY - An Acquisition Of Tele Columbus M

## 8. Write prediction files

Same format as the submitted `pred_12.csv` (`id,label`) — which is **not** overwritten.

In [10]:
for name, preds in [("pred_12_llm_v3.csv", preds_v3), ("pred_12_llm_v4.csv", preds_v4)]:
    pd.DataFrame({"id": TEST_IDS, "label": [preds[i] for i in range(N_TEST)]}).to_csv(name, index=False)
    print("wrote", name)
print("\nDone. Submitted pred_12.csv untouched.")

wrote pred_12_llm_v3.csv
wrote pred_12_llm_v4.csv

Done. Submitted pred_12.csv untouched.
